# Does the email rubric actually work?

A rubric is only useful if it **separates good outputs from bad ones**. This notebook
runs the `EmailRubric` would-send gate on five hand-written *great* cold emails and five
obviously *templated / spammy* ones, and shows the gate cleanly splits them.

**How the raw text gets scored:**
- With `ANTHROPIC_API_KEY` set, Claude scores the five dimensions (structured tool use).
- With no key, we fall back to bundled hand-scores (assigned by applying the rubric
  anchors by hand) so the notebook runs offline and reproducibly.

Either way, the **would-send decision is computed by the rubric**, never asserted. See
[`docs/CALIBRATION.md`](../docs/CALIBRATION.md) for why the thresholds sit where they do.

In [ ]:
# Run from the gtm-agent-evals/ directory (so `src` and `examples` are importable):
#   PYTHONPATH=.:src jupyter lab
from examples.email_comparison import EMAILS, evaluate_all, summarize

print(f"{len(EMAILS)} emails loaded: "
      f"{sum(e['label']=='great' for e in EMAILS)} great, "
      f"{sum(e['label']=='templated' for e in EMAILS)} templated")

## The two extremes

A *great* email leads with a specific, datable trigger and a tight, low-friction ask.
A *templated* one is mail-merge tokens, generic value props, or hype.

In [ ]:
print('GREAT  ->', EMAILS[0]['text'][:200], '...\n')
print('TEMPLATED ->', EMAILS[5]['text'][:200], '...')

## Run the gate on all ten

In [ ]:
rows = evaluate_all()
print(f"{'id':<22} {'label':<10} would_send   (scores: {rows[0].source})")
print('-' * 52)
for r in rows:
    print(f"{r.id:<22} {r.label:<10} {r.would_send}")

## Did it separate them?

In [ ]:
summary = summarize(rows)
print(f"great would-send:     {summary['great_would_send']}")
print(f"templated would-send: {summary['templated_would_send']}")
print(f"spam-risk gap (templated - great): +{summary['spam_gap']}")
print(f"cleanly separated: {summary['separated']}")

**Result (offline, illustrative scores):** 5/5 great pass, 0/5 templated pass, with a
positive spam-risk gap. The gate is an AND across all four dimensions, so an email fails
on its weakest one — a beautifully personalized email with no clear ask still doesn't ship.

Swap in the LLM scorer (set `ANTHROPIC_API_KEY`) to score real drafts from your own system;
the rubric and its thresholds don't change.